# Inference-time scaling with Qwen3-0.6B-Base

Same model, same chocolate problem — spend more compute at **decode time** and see if answers improve.

**Top 3 strategies in current use** (what we demo):

1. **Chain of Thought (CoT)** — one longer trace: “explain step by step”
2. **Self-consistency** — sample several CoT paths in parallel, **majority vote**
3. **Best-of-N** — sample several paths, **pick one** with a scorer (here: a simple length heuristic; production uses a reward / verifier model)

| Step | What we do |
| ---- | ---------- |
| 0 | Load `Qwen/Qwen3-0.6B-Base` |
| 1 | Direct ask (baseline) |
| 2 | CoT prompt |
| 3 | Self-consistency (parallel + vote) |
| 4 | Best-of-N (sample + pick) |

Run cells top to bottom. Copy useful prompts / outputs into `src/content/concepts/reasoning-models/inference-time-scaling.md`.

> Base models complete text (not chat). Tiny models are noisy — variance across runs is expected and is exactly why voting / Best-of-N help.

## 0. Setup

Use the project venv if you have it:

```bash
source .venv-qwen/bin/activate
python -m ipykernel install --user --name qwen-demo --display-name "Qwen demo"
```

Then pick kernel **Qwen demo** in this notebook.

Or install once:

```bash
pip install "torch" "transformers>=4.51" accelerate ipykernel
```

In [ ]:
from collections import Counter
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("torch", torch.__version__)
print("device will be:", "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))

In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B-Base"

PROBLEM = (
    "Ram has 3 chocolates. Sam has 5 chocolates. "
    "They put them in one basket and each eats 1. "
    "How many chocolates are left?"
)

print("Loading", MODEL_NAME, "(first run downloads ~1GB)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
)
model.eval()
print("Ready on", model.device)

### Helpers

Keep these small — easy to paste into the MD page later.

In [ ]:
def generate(
    prompt: str,
    max_new_tokens: int = 80,
    do_sample: bool = False,
    temperature: float = 0.8,
    num_return_sequences: int = 1,
):
    """Generate one or more completions for a prompt."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=0.9 if do_sample else None,
            num_return_sequences=num_return_sequences,
            pad_token_id=tokenizer.eos_token_id,
        )
    texts = []
    for out in outputs:
        full = tokenizer.decode(out, skip_special_tokens=True)
        texts.append(full[len(prompt) :].strip())
    return texts


def extract_answer(text: str):
    """Prefer 'Answer: N', else last number in the text."""
    m = re.search(r"Answer\s*:\s*(\d+)", text, re.I)
    if m:
        return m.group(1)
    nums = re.findall(r"\b(\d+)\b", text)
    return nums[-1] if nums else None


def show(title: str, prompt: str, completion: str):
    print("=" * 60)
    print(title)
    print("=" * 60)
    print("PROMPT:\n")
    print(prompt)
    print("\nCOMPLETION:\n")
    print(completion)
    print("\nPARSED ANSWER:", extract_answer(completion))
    print()

## Step 1 — Direct ask (baseline)

No reasoning instruction. Greedy decode (`do_sample=False`).

**Expectation:** often fast and sometimes wrong — no intermediate checks.

In [ ]:
prompt_direct = f"Question: {PROBLEM}\nAnswer:"

out_direct = generate(prompt_direct, max_new_tokens=40, do_sample=False)[0]
show("Step 1 — Direct", prompt_direct, out_direct)

## Step 2 — Chain of Thought (CoT)

Same model. Add **“Explain step by step”** and seed `Step 1:` so the base model continues a trace.

**Expectation:** longer completion, checkable arithmetic, often better answer.

In [ ]:
prompt_cot = (
    f"Question: {PROBLEM}\n"
    "Explain step by step, then write Answer: <number>.\n"
    "Step 1:"
)

out_cot = generate(prompt_cot, max_new_tokens=120, do_sample=False)[0]
show("Step 2 — CoT", prompt_cot, out_cot)

## Step 3 — Self-consistency (parallel strategy)

Sample **N** CoT completions (`do_sample=True`), parse each answer, take the **majority vote**.

This is the usual “parallel + sampling” combo people mean by self-consistency (Wang et al.).

**Expectation:** single bad samples get outvoted when the model is usually right.

In [ ]:
N = 5  # raise to 8–16 on GPU if you want a clearer vote

prompt_sc = (
    f"Question: {PROBLEM}\n"
    "Explain step by step, then write Answer: <number>.\n"
    "Step 1:"
)

samples = generate(
    prompt_sc,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.8,
    num_return_sequences=N,
)

answers = []
for i, text in enumerate(samples, 1):
    ans = extract_answer(text)
    answers.append(ans)
    print(f"--- sample {i} → {ans} ---")
    print(text)
    print()

votes = Counter(a for a in answers if a is not None)
majority = votes.most_common(1)[0][0] if votes else None

print("Votes:", dict(votes))
print("Majority answer:", majority)

## Step 4 — Best-of-N (sampling + select)

Same samples as Step 3, but instead of voting we **pick one** completion with a scorer.

Here the scorer is intentionally naive: **longest completion** (more steps ≈ more “effort”).  
Real systems use a process/outcome reward model or a verifier.

**Expectation:** you get one detailed trace to show; quality depends on the scorer.

In [ ]:
# Reuse `samples` from Step 3. Re-run Step 3 first if this cell errors.

def score_length(text: str) -> float:
    """Toy scorer — replace with a reward model in real setups."""
    return float(len(text))


ranked = sorted(
    ((score_length(t), extract_answer(t), t) for t in samples),
    key=lambda x: x[0],
    reverse=True,
)

print("Ranked by toy score (length):")
for rank, (score, ans, text) in enumerate(ranked, 1):
    print(f"  #{rank}  score={score:.0f}  answer={ans}")

best_score, best_ans, best_text = ranked[0]
print("\n=== Best-of-N pick ===")
print(best_text)
print("\nSelected answer:", best_ans)

## Compare (fill after you run)

Edit this cell with your numbers so you can paste a clean table into the MD file.

| Step | Strategy | Parsed answer | Notes |
| ---- | -------- | ------------- | ----- |
| 1 | Direct | ? | |
| 2 | CoT | ? | |
| 3 | Self-consistency | ? | votes: |
| 4 | Best-of-N | ? | scorer: length |

Correct answer for the chocolate problem: **6**.

### Snippet checklist for the MD page

- [ ] Model id + load cell
- [ ] `prompt_direct` + one completion
- [ ] `prompt_cot` + one completion
- [ ] 3–5 short sample answers + vote line
- [ ] Best-of-N pick (one completion)
- [ ] One-line takeaway: same weights, more decode compute → better reliability

## Optional: one-shot compare helper

Re-run all four strategies and print a compact summary (useful after you tune `N` / temperature).

In [ ]:
def run_all(n_samples: int = 5):
    direct = generate(prompt_direct, max_new_tokens=40, do_sample=False)[0]
    cot = generate(prompt_cot, max_new_tokens=120, do_sample=False)[0]
    sc_samples = generate(
        prompt_sc,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.8,
        num_return_sequences=n_samples,
    )
    sc_answers = [extract_answer(t) for t in sc_samples]
    vote = Counter(a for a in sc_answers if a).most_common(1)
    best = max(sc_samples, key=len)

    print("Direct:           ", extract_answer(direct))
    print("CoT:              ", extract_answer(cot))
    print("Self-consistency: ", vote[0][0] if vote else None, " votes=", dict(Counter(sc_answers)))
    print("Best-of-N:        ", extract_answer(best))


# Uncomment when you want a fresh summary (slow on CPU):
# run_all(5)